# 26 - Chevalley-Warning : quand le nombre de solutions d'une equation mod q est-il multiple de q ?

Serre ouvre le *Cours d'arithmetique* (ch. I) sur les corps finis, et c'est precisement le monde ou le theoreme de **Chevalley-Warning** (1935) vit : sur un corps fini, le **nombre** de solutions d'un systeme d'equations polynomiales n'est pas quelconque - il obéit a une **divisibilite** que l'on peut *compter* avant de la *prouver*.

Le geste de cette serie est de rendre l'ennoncé calculable. Ici, on ne cite pas le theoreme : on **enumere** $\mathbb{F}_q^n$, on **compte** les zeros communs, et on **verifie** l'arithmetic :

1. **Divisibilite** — si $\sum_i \deg f_i < n$, le nombre $N$ de zeros communs verifie $N \equiv 0 \pmod q$ ;
2. **Tranchance** — la borne $\sum \deg < n$ est atteinte, pas franchissable : un systeme de somme de degres $= n$ donne $N \not\equiv 0 \pmod q$ ;
3. **Corollaire de Chevalley** — une forme homogene de degre $d < n$ a un **zero non trivial** ;
4. **Second theoreme de Warning** — des que $N > 0$, on a la minoration $N \ge q^{\,n - \sum \deg}$.

Le carnet 01 en avait fait la promesse en prose (« Chevalley-Warning garantit le zero non trivial d'un polynome homogene... le monde dans lequel on calcule ici ») ; ce carnet la livre.


## 1. Le monde : $\mathbb{F}_q$ et polynomes

Un corps fini $\mathbb{F}_q$ est ici $\mathbb{Z}/q\mathbb{Z}$ (les $q$ sont premiers dans ce carnet). Un « polynome » n'est pas un objet symbolique : c'est une **fonction** $\mathbb{Z}^n \to \mathbb{Z}$ que l'on evalue modulo $q$, avec son degre total **connu par construction**. C'est tout ce qu'il faut pour compter des zeros.


In [1]:
from itertools import product

def Fq_points(q, n):
    """Itere sur tous les points de F_q^n (n-uplets d'entiers 0..q-1)."""
    return product(range(q), repeat=n)

class Poly:
    """Polynome a coefficients entiers, evalue modulo q. `degre` est le degre total (connu par construction)."""
    def __init__(self, nom, f, degre):
        self.nom, self.f, self.degre = nom, f, degre
    def est_zero(self, point, q):
        return self.f(point) % q == 0

def forme_quadratique(point):
    """Q = x1^2 + x2^2 + ... + xn^2  (degre 2)."""
    return sum(x * x for x in point)

def produit_variables(point):
    """f = x1 * x2 * ... * xn  (degre n)."""
    r = 1
    for x in point:
        r = r * x
    return r

print("Exemple : forme quadratique en 3 variables, evaluee en (1,1,1) mod 3 =",
      forme_quadratique((1, 1, 1)) % 3)
print("Exemple : produit des 2 variables en (0, 2) mod 5 =",
      produit_variables((0, 2)) % 5)


Exemple : forme quadratique en 3 variables, evaluee en (1,1,1) mod 3 = 0
Exemple : produit des 2 variables en (0, 2) mod 5 = 0


## 2. Le theoreme de Chevalley-Warning, mesure

**Theoreme (Chevalley 1935 ; Warning 1936).** Soit $\mathbb{F}_q$ un corps a $q$ elements, et $f_1, \ldots, f_k \in \mathbb{F}_q[x_1, \ldots, x_n]$ avec $\sum_{i=1}^k \deg f_i < n$. Alors le nombre $N$ de zeros communs de $f_1, \ldots, f_k$ dans $\mathbb{F}_q^n$ verifie :

$$N \equiv 0 \pmod{q} .$$

La machine de mesure est une simple enumeration : on parcourt les $q^n$ points, on compte ceux ou **tous** les polynomes s'annulent. La table ci-dessous prend une forme quadratique (degre $2$) dans $n$ variables, avec $n > 2$ : l'hypothese $\sum \deg = 2 < n$ est verifiee, et la divisibilite doit tenir sur toute la ligne.


In [2]:
def N_solutions(systeme, q, n):
    """Nombre de points de F_q^n annulant tous les polynomes de `systeme`."""
    return sum(1 for pt in Fq_points(q, n) if all(p.est_zero(pt, q) for p in systeme))

print("Divisibilite N = 0 (mod q)  —  forme quadratique Q = x1^2+...+xn^2, degre 2 < n\n")
print(f"{'q':>3} {'n':>3} {'degre':>6} {'N':>8} {'N mod q':>9}  divisible")
for q in (3, 5):
    for n in (3, 4):
        P = Poly("Q", forme_quadratique, 2)
        N = N_solutions([P], q, n)
        print(f"{q:>3} {n:>3} {2:>6} {N:>8} {N % q:>9}  {N % q == 0}")


Divisibilite N = 0 (mod q)  —  forme quadratique Q = x1^2+...+xn^2, degre 2 < n

  q   n  degre        N   N mod q  divisible
  3   3      2        9         0  True
  3   4      2       33         0  True
  5   3      2       25         0  True
  5   4      2      145         0  True


### Lecture : la divisibilite n'est pas une coïncidence

Quatre configurations, quatre corps, et **$N \bmod q = 0$ sur chaque ligne**. Ce n'est pas une propriete accidentelle de la forme quadratique : c'est l'arithmetic que le theoreme promet pour **tout** systeme satisfaisant $\sum \deg < n$. Le comptage l'atteint groupe par groupe, exactement comme le carnet 20 atteignait les theoremes de Sylow — l'ennoncé devient un predicat que le code verifie.


## 3. La borne est tranchante

Le theoreme exige $\sum \deg < n$. Cette inegalite stricte est **optimale** : des qu'elle devient une egalite, la divisibilite peut s'effondrer. Le contre-exemple canonique est $f = x_1 x_2 \cdots x_n$, de degre exactement $n$.

Son nombre de zeros s'obtient par le **complementaire** : les points ou $f$ ne s'annule pas sont ceux ou **aucune** coordonnee n'est nulle, il en est $(q-1)^n$ ; donc $N = q^n - (q-1)^n$. Reduit modulo $q$, ce comptage donne $N \equiv 0 - (-1)^n = (-1)^{n+1} \pmod q$ : **jamais nul** pour $q \ge 3$.


In [3]:
print("Tranchance : f = x1*x2*...*xn, degre = n  (hypothese somme-deg < n NON satisfaite)\n")
print(f"{'q':>3} {'n':>3} {'N':>8} {'N mod q':>9}  non-nul?")
for q in (3, 5, 7):
    for n in (2, 3):
        P = Poly("prod", produit_variables, n)   # degre n
        N = N_solutions([P], q, n)
        print(f"{q:>3} {n:>3} {N:>8} {N % q:>9}  {N % q != 0}")


Tranchance : f = x1*x2*...*xn, degre = n  (hypothese somme-deg < n NON satisfaite)

  q   n        N   N mod q  non-nul?
  3   2        5         2  True
  3   3       19         1  True
  5   2        9         4  True
  5   3       61         1  True
  7   2       13         6  True
  7   3      127         1  True


### Lecture : la borne est atteinte, pas franchissable

Sur les six lignes, $N \bmod q$ est **toujours non nul** : le theoreme ne s'applique pas (le degre vaut $n$, pas $< n$), et la divisibilite ne tient effectivement plus. La forme $q^n - (q-1)^n$ rend le mecanisme visible — ce n'est pas un contre-exemple exotique, c'est le produit de toutes les variables. L'inegalite stricte du theoreme est donc le meilleur possible.


## 4. Corollaire de Chevalley : un zero non trivial

**Corollaire.** Une forme homogene $f$ de degre $d < n$ sur $\mathbb{F}_q$ possede un **zero non trivial** (un zero autre que l'origine).

Argument : applique le theoreme a $f$ seule — $\sum \deg = d < n$, donc $N \equiv 0 \pmod q$. L'origine est toujours un zero d'une forme homogene, donc $N \ge 1$ ; comme $N$ est un multiple de $q \ge 2$, on a $N \ge q > 1$, et il reste au moins un zero non trivial. Le cas concret classique : une **forme quadratique en au moins 3 variables** sur un corps de caracteristique impaire est **isotrope**.


In [4]:
def trouve_zero_non_trivial(systeme, q, n):
    """Premier point de F_q^n (hors origine) annulant tous les polynomes, ou None."""
    for pt in Fq_points(q, n):
        if any(x != 0 for x in pt) and all(p.est_zero(pt, q) for p in systeme):
            return pt
    return None

print("Corollaire de Chevalley : zero non trivial d'une forme quadratique (degre 2 < n)\n")
for q in (3, 5, 7):
    for n in (3, 4):
        z = trouve_zero_non_trivial([Poly("Q", forme_quadratique, 2)], q, n)
        print(f"q={q} n={n}:  zero non trivial = {z}")


Corollaire de Chevalley : zero non trivial d'une forme quadratique (degre 2 < n)

q=3 n=3:  zero non trivial = (1, 1, 1)
q=3 n=4:  zero non trivial = (0, 1, 1, 1)
q=5 n=3:  zero non trivial = (0, 1, 2)
q=5 n=4:  zero non trivial = (0, 0, 1, 2)
q=7 n=3:  zero non trivial = (1, 2, 3)
q=7 n=4:  zero non trivial = (0, 1, 2, 3)


### Lecture : l'existence devient une recherche terminée

Chaque configuration renvoie un zero concret, trouve par un parcours qui **termine** (l'espace est fini). Le plus parlant est $q=3, n=3$ : les carres modulo 3 valent $0$ ou $1$, donc $x_1^2+x_2^2+x_3^2 \equiv 0 \pmod 3$ exige soit tout nul, soit les trois carres vaudre $1$ — d'ou $(1,1,1)$. En deux variables, la meme forme n'a **que** le zero trivial sur $\mathbb{F}_3$ : c'est le frontier $d = n$ que la section 3 a rendu tranchante.


## 5. Second theoreme de Warning : une minoration

**Second theoreme de Warning (1936).** Si, en plus de $\sum \deg f_i < n$, le systeme admet au moins un zero ($N > 0$), alors

$$N \ge q^{\,n - \sum \deg f_i} .$$

Le theoreme 1 dit que $N$ est un multiple de $q$ ; le theoreme 2 **mine** ce multiple. Pour une forme homogene de degre $d < n$ (qui a toujours l'origine comme zero, d'ou $N>0$), la borne est $q^{\,n-d}$.


In [5]:
print("Second theoreme de Warning : N >= q^(n - degre), forme homogene de degre 2 < n\n")
print(f"{'q':>3} {'n':>3} {'degre d':>9} {'N':>6} {'q^(n-d)':>9}  borne tenue?")
for q in (3, 5):
    for n in (3, 4):
        d = 2
        P = Poly("Q", forme_quadratique, d)
        N = N_solutions([P], q, n)
        borne = q ** (n - d)
        print(f"{q:>3} {n:>3} {d:>9} {N:>6} {borne:>9}  {N >= borne}")


Second theoreme de Warning : N >= q^(n - degre), forme homogene de degre 2 < n

  q   n   degre d      N   q^(n-d)  borne tenue?
  3   3         2      9         3  True
  3   4         2     33         9  True
  5   3         2     25         5  True
  5   4         2    145        25  True


### Lecture : le multiple a une taille

La borne $q^{\,n-d}$ est tenue sur chaque ligne, avec une **marge** : $q=3, n=4$ donne $N = 33$ contre une borne $3^{2} = 9$. Le second theoreme ne dit pas seulement que $N$ est multiple de $q$ — il dit que ce multiple est **grand**. (Le raffinement d'Ax-Katz, qui précise l'exposant exact de la divisibilite, fait l'objet de l'exercice 3.)


## Exercices


### Exercice 1 - La machine, a vous de l'ecrire

Ecrivez `compter(systeme, q, n)` qui renvoie le nombre de points de $\mathbb{F}_q^n$ annulant tous les polynomes de `systeme` **sans** reutiliser `N_solutions` : re-ecrivez l'enumeration a la main, c'est l'objet de l'exercice. Puis, pour $q=3$ et $n$ de $3$ a $5$, affichez $N$ et $N \bmod q$ pour la forme quadratique : la divisibilite doit tenir sur toute la ligne.

`# Indice` : `product(range(q), repeat=n)` genere les points ; un point est zero s'il annule `all` les polynomes.


In [6]:
def compter(systeme, q, n):
    """Nombre de points de F_q^n annulant tous les polynomes de systeme."""
    # TODO etudiant : eumerer les points et compter ceux qui annulent tout systeme
    return None  # TODO etudiant

print("Exercice 1 a completer : compter([], 3, 3) renvoie", compter([], 3, 3))


Exercice 1 a completer : compter([], 3, 3) renvoie

 None


### Exercice 2 - La borne tranchante, par le complementaire

Complétez `troncature(q, n)` : pour $f = x_1 x_2 \cdots x_n$ (degre $n$), calculez $N$ par la formule du **complementaire** $N = q^n - (q-1)^n$ et renvoyez $N \bmod q$. Verifiez sur $(q,n) \in \{(3,2),(5,2),(3,3),(5,3),(7,3)\}$ que le resultat est **toujours non nul**.

`# Indice` : « $f$ ne s'annule pas » equivaut a « aucune coordonnee n'est nulle », il y a $(q-1)$ choix par coordonnee.


In [7]:
def troncature(q, n):
    """N mod q pour f = x1*...*xn, par le compte du complementaire."""
    # TODO etudiant : N = q**n - (q-1)**n ; renvoyer N % q
    return None  # TODO etudiant

print("Exercice 2 a completer : troncature(3, 2) renvoie", troncature(3, 2))


Exercice 2 a completer : troncature(3, 2) renvoie None


### Exercice 3 - La minoration, et l'ecart

Complétez `borne_warning(q, n, d)` : elle renvoie la borne inferieure $q^{\,n-d}$ du second theoreme de Warning pour une forme homogene de degre $d < n$. Puis, pour $q=3$ et $n=5$, comparez cette borne a l'effectif $N$ de la forme quadratique (degre $2$) et affichez l'ecart $N - q^{\,n-d}$ : il doit etre **positif**.

`# Indice` : la borne est une simple puissance ; l'effectif se compte avec la machine de la section 2.


In [8]:
def borne_warning(q, n, d):
    """Borne inferieure q^(n-d) du second theoreme de Warning."""
    # TODO etudiant : renvoyer q ** (n - d)
    return None  # TODO etudiant

print("Exercice 3 a completer : borne_warning(3, 5, 2) renvoie", borne_warning(3, 5, 2))


Exercice 3 a completer : borne_warning(3, 5, 2) renvoie None


## Conclusion et limites honnetes

Ce que le carnet etablit, et par quel moyen :

- la **divisibilite** $N \equiv 0 \pmod q$ est **mesuree** sur une famille de formes quadratiques (degre $2 < n$), corps apres corps ;
- la **tranchance** de la borne $\sum \deg < n$ est **exhibite** par $f = x_1\cdots x_n$, dont le comptage par le complementaire donne $N \equiv (-1)^{n+1} \not\equiv 0 \pmod q$ ;
- le **corollaire de Chevalley** (zero non trivial) et le **second theoreme de Warning** (minoration $q^{\,n-d}$) sont **verifies** sur les memes configurations.

Ce que le carnet **ne** fait pas, et qu'il faut dire :

1. **Aucune preuve** n'est produite ici : ce sont des verifications sur des cas, et le passage au general est le role du theoreme (Chevalley 1935 ; Warning 1936), dont la preuve utilise l'identite de Freshman's Dream $\sum_{x \in \mathbb{F}_q} x^m = 0$ — le versant Lean de la serie le portera, comme pour Sylow (carnet 20).
2. **Les corps sont premiers** : le comptage vit sur $\mathbb{Z}/q\mathbb{Z}$ ; les corps $\mathbb{F}_{p^k}$ avec $k>1$ demanderaient une representation par polynomes irreducibles, hors du cadre stdlib de cette tranche.
3. **La borne de calcul est reelle** : l'enumeration coute $O(q^n)$, tenable ici pour $q^n \lesssim 3000$ ; au-dela, il faudrait compter sans eumerer (le principe d'inclusion-exclusion, ou la preuve elle-meme).


## Ressources

- J.-P. Serre, *Cours d'arithmetique*, PUF, 1970 — ch. I (corps finis) : le monde ou vivent les theoremes de Chevalley et de Warning.
- J.-P. Serre, *Plaisir des mathematiques* (IHP, 2026, transcription au gisement) : la defense du contre-exemple comme theoreme, dont le carnet 01 a fait le fil conducteur.
- Chevalley, *Demonstration d'une hypothese d'Artin* (1935) ; Warning, *Bemerkung zur vorstehenden Arbeit* (1936) : les theoremes 1 et 2 distilles ici.
- Carnet 01 de cette serie (corps finis, borne de Hasse) : la promesse en prose que ce carnet livre ; carnet 20 (Sylow) : le meme geste « l'ennoncé devient un comptage ».
